# 14.5 Stacks and Queues

**Prerequisites:** 14.2 Python's Built-ins, 14.4 Linked Lists  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- **LIFO vs FIFO** - and how to tell instantly which a problem wants
- Stacks with a plain `list`; queues with `deque`
- 🔴 Why `list` as a queue is a performance bug (**14.2**, measured again)
- Bracket matching, expression evaluation, and the call stack itself
- **Monotonic stacks** - the pattern behind a whole family of O(n) solutions
- **Min stack** - O(1) minimum, which sounds impossible
- Building a queue from two stacks, and why it is amortised O(1)
- `deque` vs `queue.Queue` - and when thread safety matters (**12.2**)
- Interview questions, worked

---

## Two disciplines for adding and removing

Both are about **which element you are allowed to take next**.

```
   STACK  (LIFO)                    QUEUE  (FIFO)
   last in, first out               first in, first out

        push │  ▲ pop                enqueue        dequeue
             ▼  │                        │             ▲
           ┌────────┐                    ▼             │
           │   C    │  <- top       ┌────┬────┬────┐
           │   B    │               │ A  │ B  │ C  │
           │   A    │               └────┴────┴────┘
           └────────┘                 ^front      back

   a stack of plates                 a queue at a counter
```

| | Stack | Queue |
|---|---|---|
| Add | `push` — to the top | `enqueue` — to the back |
| Remove | `pop` — from the top | `dequeue` — from the front |
| Python | `list` | `collections.deque` |
| Both operations | O(1) | O(1) |

### How to recognise which you need

> **Stack** when the problem involves **nesting, matching, undoing, or backtracking** — anything where the most recent thing must be resolved first.
>
> **Queue** when the problem involves **order of arrival, or exploring level by level** — scheduling, buffering, breadth-first search (**14.9**).

In [ ]:
from collections import deque

# ---- a stack is just a list ----
stack = []
for item in ("A", "B", "C"):
    stack.append(item)               # push - O(1) amortised (14.1)
print("stack       :", stack, " top is the RIGHT end")
print("  peek      :", stack[-1])
print("  pop       :", stack.pop())  # O(1)
print("  after pop :", stack)
print("  empty?    :", not stack)    # the Pythonic emptiness test

# ---- a queue is a deque ----
queue = deque()
for item in ("A", "B", "C"):
    queue.append(item)               # enqueue at the back - O(1)
print("\nqueue       :", queue)
print("  peek front:", queue[0])
print("  dequeue   :", queue.popleft())   # O(1) - the whole point
print("  after     :", queue)

# ---- a deque is both, and more ----
both = deque([1, 2, 3])
both.appendleft(0)
both.append(4)
print("\ndeque as both ends:", both)
print("  popleft:", both.popleft(), " pop:", both.pop(), "->", both)

### 🔴 The mistake worth measuring twice

**14.2** covered this and it belongs here too, because "implement a queue" is where people reach for a list.

```
    queue = []
    queue.append(x)      O(1)  ✅
    queue.pop(0)         O(n)  🔴  shifts every remaining element
```

Appending is fine, so the bug hides. It is the *removal* that makes the whole loop O(n²).

In [ ]:
import time

SIZE = 25_000

as_list = list(range(SIZE))
started = time.perf_counter()
while as_list:
    as_list.pop(0)                   # 🔴 O(n) each time
list_time = time.perf_counter() - started

as_deque = deque(range(SIZE))
started = time.perf_counter()
while as_deque:
    as_deque.popleft()               # ✅ O(1)
deque_time = time.perf_counter() - started

print(f"draining {SIZE:,} items as a queue\n")
print(f"  list.pop(0)      {list_time * 1000:9.1f} ms   O(n) each -> O(n^2) total")
print(f"  deque.popleft()  {deque_time * 1000:9.1f} ms   O(1) each -> O(n) total")
print(f"  ratio            {list_time / deque_time:9,.0f}x")
print("\n  A list is the right stack and the wrong queue.")

---

# Stacks in practice

## 1. Matching brackets

The canonical stack problem, and the shape of every parser you will ever meet.

```
   {[()]}     push {   push [   push (   pop ( matches )   pop [ matches ]   pop {
   {[(])}     ...      ...      push (   pop ( vs ]  ->  MISMATCH
```

**Why a stack:** the most recently opened bracket must be the first one closed. That sentence *is* LIFO.

Three ways to fail, and all three must be handled:

| Failure | Detect |
|---|---|
| wrong closer | top of stack does not match |
| closer with nothing open | stack is empty when a closer arrives |
| unclosed opener | stack is **not empty** at the end |

In [ ]:
PAIRS = {")": "(", "]": "[", "}": "{"}


def brackets_balanced(text):
    """O(n) time, O(n) space in the worst case (all openers)."""
    stack = []
    for char in text:
        if char in "([{":
            stack.append(char)
        elif char in PAIRS:
            if not stack or stack.pop() != PAIRS[char]:
                return False            # wrong closer, or nothing was open
    return not stack                    # anything left open is unbalanced


for sample in ("{[()]}", "{[(])}", "(((", ")", "", "a(b)c[d]"):
    print(f"  {sample!r:<12} {brackets_balanced(sample)}")

print("\n🔴 The three failure modes, each needing its own check:")
print("   '{[(])}'  wrong closer       -> stack.pop() != expected")
print("   ')'       nothing was open   -> `not stack`")
print("   '((('     never closed       -> `return not stack` at the end")
print("\n   Forgetting the last one is the most common bug: '(((' would")
print("   otherwise be reported as balanced.")

## 2. Evaluating expressions

**Reverse Polish Notation** puts the operator *after* its operands: `3 4 +` means `3 + 4`. No brackets are needed, and evaluation is a simple stack walk:

```
   "3 4 + 2 *"        stack
     3                 [3]
     4                 [3, 4]
     +                 [7]        pop 4, pop 3, push 3+4
     2                 [7, 2]
     *                 [14]       pop 2, pop 7, push 7*2
```

🔴 **Operand order matters.** The stack gives you the *second* operand first, so `right = pop()` then `left = pop()`. Getting this backwards is invisible for `+` and `*` and wrong for `-` and `/` — which is exactly the kind of bug that survives testing.

In [ ]:
import operator

OPERATORS = {
    "+": operator.add,
    "-": operator.sub,
    "*": operator.mul,
    "/": operator.truediv,
}


def evaluate_rpn(expression):
    """Evaluate reverse Polish notation. O(n) time and space."""
    stack = []
    for token in expression.split():
        if token in OPERATORS:
            if len(stack) < 2:
                raise ValueError(f"not enough operands for {token!r}")
            right = stack.pop()          # 🔴 the SECOND operand comes off first
            left = stack.pop()
            stack.append(OPERATORS[token](left, right))
        else:
            stack.append(float(token))
    if len(stack) != 1:
        raise ValueError("malformed expression")
    return stack[0]


for expression in ("3 4 +", "3 4 + 2 *", "5 1 2 + 4 * + 3 -", "10 2 /", "2 10 /"):
    print(f"  {expression:<20} = {evaluate_rpn(expression)}")

print("\n  '10 2 /' is 5.0 and '2 10 /' is 0.2 - the order check that catches")
print("  a reversed pop. With only + and * you would never notice.")

for bad in ("3 +", "3 4"):
    try:
        evaluate_rpn(bad)
    except ValueError as exc:
        print(f"  {bad!r:<8} -> ValueError: {exc}")

## 3. The call stack is a stack

Every function call pushes a **frame** holding its local variables and where to return to. Returning pops it. That is why:

- recursion depth costs memory — O(depth) (**14.1**)
- too much recursion raises `RecursionError` — the stack is finite
- a traceback reads bottom-up — it is literally the stack, printed

**Any recursive algorithm can be rewritten iteratively with an explicit stack.** Sometimes that is worth doing, to escape the recursion limit; it is how you traverse a very deep tree (**14.7**).

In [ ]:
import inspect


def outer():
    return middle()


def middle():
    return inner()


def inner():
    return [frame.function for frame in inspect.stack()[:4]]


print("the live call stack, innermost first:")
print("  ", outer())


# The same computation, recursive and with an explicit stack.
def countdown_recursive(n, out):
    if n == 0:
        return out
    out.append(n)
    return countdown_recursive(n - 1, out)


def countdown_iterative(n):
    out, stack = [], [n]
    while stack:
        current = stack.pop()
        if current > 0:
            out.append(current)
            stack.append(current - 1)
    return out


print("\nrecursive to 5   :", countdown_recursive(5, []))
print("explicit stack   :", countdown_iterative(5))

try:
    countdown_recursive(5_000, [])
    print("\nrecursive to 5,000: fine")
except RecursionError:
    print("\nrecursive to 5,000: RecursionError")
print("explicit stack to 5,000:", len(countdown_iterative(5_000)), "items - no limit")
print("\n  The explicit stack IS the call stack, moved onto the heap.")

---

# 🔴 Monotonic stacks

A stack kept deliberately sorted — always increasing, or always decreasing. Before pushing, pop everything that would break the order.

This one idea turns a family of O(n²) problems into O(n): *next greater element*, *daily temperatures*, *largest rectangle in a histogram*, *stock span*.

```
   next greater element for [2, 1, 2, 4, 3]

   see 2   stack empty            push index 0        stack [2]
   see 1   1 < 2, keeps order     push index 1        stack [2, 1]
   see 2   2 > 1  -> answer for 1 is 2, pop           stack [2]
           2 == 2, keeps order    push index 2        stack [2, 2]
   see 4   4 > 2 -> answer, pop; 4 > 2 -> answer, pop stack []
                                  push index 3        stack [4]
   see 3   3 < 4                  push index 4        stack [4, 3]
```

### Why it is O(n), not O(n²)

> The inner `while` looks like it makes this quadratic. It does not: **each element is pushed exactly once and popped at most once**, so there are at most 2n stack operations in total.

That is the same amortised argument as the sliding window in **14.3** — and interviewers ask for it explicitly.

In [ ]:
def next_greater(data):
    """For each element, the next element to its right that is larger.

    -1 where there is none. O(n) time, O(n) space.
    """
    result = [-1] * len(data)
    stack = []                          # holds INDICES, not values
    operations = 0
    for index, value in enumerate(data):
        while stack and data[stack[-1]] < value:
            result[stack.pop()] = value
            operations += 1
        stack.append(index)
        operations += 1
    return result, operations


def next_greater_brute(data):
    result = [-1] * len(data)
    operations = 0
    for i in range(len(data)):
        for j in range(i + 1, len(data)):
            operations += 1
            if data[j] > data[i]:
                result[i] = data[j]
                break
    return result, operations


sample = [2, 1, 2, 4, 3]
fast, _ = next_greater(sample)
slow, _ = next_greater_brute(sample)
print(f"  {sample} -> {fast}   (brute force agrees: {fast == slow})")

# the worst case for brute force: strictly decreasing, so nothing is ever found
descending = list(range(2_000, 0, -1))
_, fast_ops = next_greater(descending)
_, slow_ops = next_greater_brute(descending)
print(f"\n  n = {len(descending):,}, strictly decreasing:")
print(f"    monotonic stack : {fast_ops:>10,} operations   O(n)")
print(f"    brute force     : {slow_ops:>10,} operations   O(n^2)")
print(f"    ratio           : {slow_ops / fast_ops:>10,.0f}x")
print("\n  Each index was pushed once and popped at most once: <= 2n.")

In [ ]:
def daily_temperatures(temperatures):
    """How many days until a warmer day? The same pattern, storing distance."""
    result = [0] * len(temperatures)
    stack = []
    for day, temp in enumerate(temperatures):
        while stack and temperatures[stack[-1]] < temp:
            earlier = stack.pop()
            result[earlier] = day - earlier      # distance, not value
        stack.append(day)
    return result


temps = [73, 74, 75, 71, 69, 72, 76, 73]
print("temperatures :", temps)
print("days to warmer:", daily_temperatures(temps))
print("\n  0 means no warmer day follows. Same six lines as next_greater -")
print("  only what gets stored changed.")


def largest_rectangle(heights):
    """Largest rectangle in a histogram. The hardest classic monotonic-stack
    problem, and worth studying: a sentinel 0 flushes the stack at the end.
    """
    stack = []
    best = 0
    for index, height in enumerate(heights + [0]):     # sentinel
        while stack and heights[stack[-1]] >= height:
            tall = heights[stack.pop()]
            left = stack[-1] + 1 if stack else 0
            best = max(best, tall * (index - left))
        stack.append(index)
    return best


print()
for bars in ([2, 1, 5, 6, 2, 3], [2, 4], [5], [1, 1, 1, 1]):
    print(f"  largest rectangle in {str(bars):<18} = {largest_rectangle(bars)}")
print("\n  The trailing 0 is a sentinel: it is smaller than everything, so")
print("  it forces the stack to empty and every remaining bar to be measured.")

## Min stack - O(1) minimum

*"Design a stack that also reports its minimum in O(1)."*

Scanning for the minimum is O(n). Keeping a single `min` variable fails, because popping the minimum leaves you with no idea what the new one is.

**The trick:** a second stack holding the minimum *as it was* at each level. Push to it on every push; pop from it on every pop. It always mirrors the main stack's history.

```
   push 5   main [5]        mins [5]
   push 3   main [5,3]      mins [5,3]
   push 7   main [5,3,7]    mins [5,3,3]   <- 7 is not smaller; repeat 3
   pop      main [5,3]      mins [5,3]     -> min is 3
   pop      main [5]        mins [5]       -> min is 5, recovered for free
```

This is a **space-for-time trade** (**14.1**): O(n) extra memory buys O(1) queries.

In [ ]:
class MinStack:
    """All operations O(1). Extra space O(n)."""

    def __init__(self):
        self._values = []
        self._minimums = []             # the minimum as of each level

    def push(self, value):
        self._values.append(value)
        smallest = value if not self._minimums else min(value, self._minimums[-1])
        self._minimums.append(smallest)

    def pop(self):
        if not self._values:
            raise IndexError("pop from empty stack")
        self._minimums.pop()
        return self._values.pop()

    def top(self):
        return self._values[-1]

    def minimum(self):
        if not self._minimums:
            raise IndexError("minimum of empty stack")
        return self._minimums[-1]       # O(1)

    def __len__(self):
        return len(self._values)


stack = MinStack()
for value in (5, 3, 7, 3, 8):
    stack.push(value)
    print(f"  push {value} -> top {stack.top()}, min {stack.minimum()}")

print()
while len(stack):
    top = stack.top()
    smallest = stack.minimum()
    stack.pop()
    remaining = stack.minimum() if len(stack) else "-"
    print(f"  pop {top} (min was {smallest}) -> min now {remaining}")

print("\n  The minimum is recovered exactly, with no scanning, because the")
print("  second stack recorded the history rather than just the answer.")

## A queue from two stacks

A classic, and a genuinely elegant amortised argument.

```
   inbox (push here)      outbox (pop here)

   enqueue: push onto the inbox                       O(1)
   dequeue: if the outbox is empty, tip the ENTIRE
            inbox into it - which reverses the order  O(n) that one time
            then pop from the outbox                  O(1)
```

**Why it is amortised O(1):** each element is moved from inbox to outbox **exactly once** in its lifetime. Over n operations the total transfer work is O(n), so the average per operation is O(1) — the same reasoning as `list.append` in **14.1**.

🔴 The common bug is tipping the inbox across on *every* dequeue. Only do it when the outbox is empty, or the whole argument collapses and it really is O(n) each time.

In [ ]:
class QueueFromStacks:
    """Amortised O(1) per operation."""

    def __init__(self):
        self._inbox = []
        self._outbox = []
        self.transfers = 0              # instrumentation, to prove the claim

    def enqueue(self, value):
        self._inbox.append(value)

    def _tip(self):
        # 🔴 ONLY when the outbox is empty
        if not self._outbox:
            while self._inbox:
                self._outbox.append(self._inbox.pop())
                self.transfers += 1

    def dequeue(self):
        self._tip()
        if not self._outbox:
            raise IndexError("dequeue from empty queue")
        return self._outbox.pop()

    def __len__(self):
        return len(self._inbox) + len(self._outbox)


queue = QueueFromStacks()
for value in ("A", "B", "C"):
    queue.enqueue(value)
print("  dequeue:", queue.dequeue(), queue.dequeue())
queue.enqueue("D")
print("  dequeue:", queue.dequeue(), queue.dequeue())
print("  FIFO order preserved\n")

# prove the amortised claim
N = 10_000
queue = QueueFromStacks()
for i in range(N):
    queue.enqueue(i)
for _ in range(N):
    queue.dequeue()
print(f"  {N:,} enqueues + {N:,} dequeues")
print(f"  total element transfers: {queue.transfers:,}")
print(f"  transfers per element  : {queue.transfers / N:.1f}")
print("\n  Exactly one transfer each. That is the amortised O(1) argument,")
print("  measured rather than asserted.")

## `deque` vs `queue.Queue`

Both are called queues; they solve different problems.

| | `collections.deque` | `queue.Queue` |
|---|---|---|
| Purpose | a **data structure** | a **thread communication** primitive |
| Thread-safe | append/pop are, individually | **fully**, by design |
| Blocking `get()` | no | **yes**, with an optional timeout |
| `task_done()`/`join()` | no | yes |
| Speed | faster | slower - locking costs |

> **Choose by intent.** Algorithm work — BFS, sliding windows, undo — wants `deque`. Handing work between threads wants `queue.Queue`, whose blocking `get()` and `task_done()` are exactly what a worker pool needs (**12.2**).

🔴 `deque` being "thread-safe" only covers *individual* operations. `if d: d.popleft()` is two operations and can still race — another thread can empty it in between.

In [ ]:
import queue

# ---- deque: fast, and bounded with maxlen ----
recent = deque(maxlen=3)                # a fixed-size sliding window
for event in ("login", "view", "click", "logout", "login"):
    recent.append(event)
    print(f"  after {event:<8} -> {list(recent)}")
print("\n  maxlen makes it a ring buffer: the oldest is dropped silently.")
print("  Ideal for 'the last N events' with no bookkeeping.")

# ---- queue.Queue: blocking, for threads ----
work = queue.Queue(maxsize=2)
work.put("job-1")
work.put("job-2")
print(f"\n  queue.Queue full: {work.full()}")
try:
    work.put("job-3", timeout=0.1)      # would block; times out instead
except queue.Full:
    print("  put() raised queue.Full - back-pressure, by design")
print("  get():", work.get())

empty = queue.Queue()
try:
    empty.get(timeout=0.1)
except queue.Empty:
    print("  get() on an empty queue raised queue.Empty after the timeout")
print("\n  Blocking and back-pressure are what deque does NOT give you (12.2).")

## Interview questions

**1. Valid parentheses.** *(implemented above)*
> Stack. Remember all three failure modes — especially the non-empty stack at the end.

**2. Min stack — O(1) minimum.** *(implemented above)*
> A second stack of running minimums. Space for time.

**3. Implement a queue using two stacks.** *(implemented above)*
> Amortised O(1). The key sentence is *each element moves between the stacks exactly once*.

**4. Implement a stack using two queues.**
> The mirror image, and strictly worse: one of push or pop must be O(n). Say which you chose to make expensive and why.

**5. Next greater element / daily temperatures.** *(implemented above)*
> Monotonic stack, O(n). Volunteer the amortised argument before being asked.

**6. Largest rectangle in a histogram.** *(implemented above)*
> Monotonic stack with a sentinel. One of the hardest of the family — if you can explain the sentinel, you understand the pattern.

**7. Evaluate reverse Polish notation.** *(implemented above)*
> Stack. Watch the operand order for `-` and `/`.

**8. Sliding window maximum.**
> A **monotonic deque** holding indices in decreasing order of value: O(n). The heap solution is O(n log n) — mention both and say which is better.

**9. Design a browser history (back/forward).**
> Two stacks. Visiting a new page clears the forward stack — that detail is the whole question.

**10. When would you use `deque` over `list`?**
> Whenever you need the front: `popleft`/`appendleft` are O(1) versus O(n). Never for indexed access, which is O(n) on a deque (**14.2**).

In [ ]:
# Question 8, because it combines both structures in this notebook.
def sliding_window_max(data, k):
    """Maximum of every window of size k. O(n) with a monotonic deque.

    The deque holds INDICES, with their values in decreasing order, so the
    front is always the maximum of the current window.
    """
    if k <= 0 or not data:
        return []
    window = deque()
    result = []
    for index, value in enumerate(data):
        # drop indices that have fallen out of the window on the left
        while window and window[0] <= index - k:
            window.popleft()
        # drop values smaller than the incoming one: they can never win again
        while window and data[window[-1]] < value:
            window.pop()
        window.append(index)
        if index >= k - 1:
            result.append(data[window[0]])
    return result


def sliding_window_max_brute(data, k):
    return [max(data[i:i + k]) for i in range(len(data) - k + 1)]


cases = [([1, 3, -1, -3, 5, 3, 6, 7], 3), ([9, 8, 7], 2), ([1], 1), ([4, 2], 2)]
for values, k in cases:
    fast = sliding_window_max(values, k)
    slow = sliding_window_max_brute(values, k)
    print(f"  k={k}  {str(values):<26} -> {str(fast):<22} "
          f"agrees: {fast == slow}")

print("\n  Two `while` loops inside a `for`, and still O(n): every index is")
print("  appended once and removed once. The same argument as always.")
print("\n  The brute-force version is O(n*k) - it recomputes max() per window,")
print("  which is exactly what 14.3 taught us to stop doing.")

---

## Common Mistakes & Pitfalls

1. 🔴 **Using a `list` as a queue.** `pop(0)` is O(n); the loop becomes O(n²). Use `deque`.
2. 🔴 **Forgetting to check the stack is empty at the end** of bracket matching. `'((('` is otherwise reported as balanced.
3. 🔴 **Reversing the operand order in RPN.** Invisible for `+` and `*`, wrong for `-` and `/`.
4. 🔴 **Tipping the inbox on every dequeue** in the two-stack queue. Only when the outbox is empty, or it is O(n) each time.
5. **Popping from an empty stack.** `IndexError`. Check `if stack:` first.
6. **Storing values instead of indices in a monotonic stack.** You usually need the position to compute a distance or width.
7. **Assuming a nested `while` makes it O(n²).** Count total pushes and pops - each element moves at most twice.
8. **Reaching for `queue.Queue` in single-threaded code.** It is slower and you gain nothing; use `deque` (**12.2**).
9. **Treating `deque` as fully thread-safe.** Individual operations are atomic; check-then-act sequences are not.

## Best Practices

- `list` for stacks, `deque` for queues - and know why.
- Use `deque(maxlen=n)` for a fixed-size window; it drops the oldest for you.
- Push indices, not values, when you may need position or distance.
- State the amortised argument whenever a `while` sits inside a `for`.
- Test empty input, one element, and all-equal elements.
- Use `not stack` rather than `len(stack) == 0`.
- Reach for a monotonic stack whenever the question asks for the *next* or *previous* greater or smaller element.
- Use `queue.Queue` only for cross-thread handoff, where its blocking is the point.

## Practice Exercises

Try these before moving on.

1. Extend `brackets_balanced` to report the *index* of the first mismatch rather than just `False`.
2. Implement a stack using two queues. Which operation did you make O(n), and why is that the better choice?
3. 🔴 Implement `previous_smaller_element` with a monotonic stack, then verify it against brute force on 100 random arrays.
4. Implement browser history with two stacks, supporting `visit`, `back` and `forward`. What must `visit` do to the forward stack?
5. Convert an infix expression such as `3 + 4 * 2` to RPN using the shunting-yard algorithm - a stack for operators, output for operands.
6. Extend `MinStack` with an O(1) `maximum()` as well. How much extra space does that cost?
7. Implement `sliding_window_min` by changing one comparison, and confirm it against brute force.
8. 🔴 Time `sliding_window_max` against the brute-force version for n=100,000 and k=1,000. Predict the ratio from the complexities first, then measure (**14.1**).